<a href="https://colab.research.google.com/github/javageek2018/AirlineArrivalDelay/blob/Modeling/Flights_GradientBoosting_Angela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [2]:
from pyspark.sql.functions import col
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
import shutil, os, glob

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from xgboost.spark import SparkXGBClassifier

# Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Set Up Spark

In [4]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [5]:
# spark = SparkSession.builder \
#     .appName("ColabSpark") \
#     .master("local[*]") \
#     .config("spark.driver.memory", "8g") \
#     .getOrCreate()

# spark


spark = SparkSession.builder \
    .appName("FlightDelayXGB") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark

# Read Data (Spark)

In [6]:
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

In [7]:
base_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded"

train_file_path = f"{base_path}/train.parquet"
validate_file_path = f"{base_path}/val.parquet"
test_file_path = f"{base_path}/test.parquet"


df_train = spark.read.parquet(train_file_path)
df_val = spark.read.parquet(validate_file_path)
df_test = spark.read.parquet(test_file_path)

In [8]:
print(f"Train: {df_train.count():,} rows")
print(f"Val  : {df_val.count():,} rows")
print(f"Test : {df_test.count():,} rows")

Train: 31,149,502 rows
Val  : 6,743,403 rows
Test : 6,965,246 rows


## Sample the Data

In [9]:
print("📅 Year ranges:")
df_train.select(F.min('Year'), F.max('Year')).show()
df_val.select(F.min('Year'), F.max('Year')).show()
df_test.select(F.min('Year'), F.max('Year')).show()

📅 Year ranges:
+---------+---------+
|min(Year)|max(Year)|
+---------+---------+
|     2018|     2022|
+---------+---------+

+---------+---------+
|min(Year)|max(Year)|
+---------+---------+
|     2023|     2023|
+---------+---------+

+---------+---------+
|min(Year)|max(Year)|
+---------+---------+
|     2024|     2024|
+---------+---------+



In [10]:
SAMPLE_FRACTION = 0.10   # tune based on Colab RAM

# Build stratification key: Year × Month × Class
df_train_keyed = df_train.withColumn(
    'strat_key',
    F.concat_ws('_',
        F.col('Year').cast('string'),
        F.col('Month').cast('string'),
        F.col('ArrDel15').cast('string')
    )
)

# Get all unique strata (5 years × 12 months × 2 classes = 120 strata)
strata = [row['strat_key'] for row in
          df_train_keyed.select('strat_key').distinct().collect()]

print(f"Number of strata: {len(strata)}")

# Same fraction for every stratum
fractions = {s: SAMPLE_FRACTION for s in strata}

# Sample
df_train_sampled = (
    df_train_keyed
    .sampleBy('strat_key', fractions=fractions, seed=42)
    .drop('strat_key')
)

Number of strata: 120


In [11]:
df_train_sampled = df_train_sampled.cache()
df_val_full      = df_val.cache()         # keep val full
df_test_full     = df_test.cache()        # keep test full

# Force materialization (also gives you counts)
train_n = df_train_sampled.count()
val_n   = df_val_full.count()
test_n  = df_test_full.count()

print(f"Train (sampled): {train_n:,}")
print(f"Val   (full 2023): {val_n:,}")
print(f"Test  (full 2024): {test_n:,}")

Train (sampled): 3,115,085
Val   (full 2023): 6,743,403
Test  (full 2024): 6,965,246


In [12]:
# Sample val to ~1M for fast early-stopping evaluation
df_val_sampled = df_val.sampleBy(
    'ArrDel15',
    fractions={0: 0.15, 1: 0.15},   # ~15% → ~1M rows
    seed=42
).cache()

print(f"Val sampled: {df_val_sampled.count():,}")

Val sampled: 1,012,424


In [13]:
df_train.head()

Row(Year=2018, Quarter=1, Month=1, DayofMonth=1, DayOfWeek=1, FlightDate=1514764800000000000, Reporting_Airline='DL', Flight_Number_Reporting_Airline='849.0', Origin='MSY', Dest='ATL', CRSDepTime=600, DepTimeBlk='0600-0659', CRSArrTime=832, ArrDel15=0, CRSElapsedTime=92.0, Distance=425.0, DistanceGroup=2, date='2018-01-01', dep_hour=6, arr_hour=8, dep_hour_minus2=4, arr_hour_minus2=6, origin_temp_f=28.9, origin_dewpoint_f=16.0, origin_humidity=58.06, origin_feels_like_f=17.6, origin_wind_kts=12.583333333333334, origin_gust_kts=25.0, origin_visibility=10.0, origin_precip_in=0.0, origin_wx_codes='none', origin_is_rain=0, origin_is_snow=0, origin_is_fog=0, origin_low_visibility=0, origin_high_wind=0, origin_severe_weather=0, dest_temp_f=19.0, dest_dewpoint_f=5.0, dest_humidity=53.8, dest_feels_like_f=6.04, dest_wind_kts=12.384615384615385, dest_gust_kts=16.0, dest_visibility=10.0, dest_precip_in=0.0, dest_wx_codes='none', dest_is_rain=0, dest_is_snow=0, dest_is_fog=0, dest_low_visibility=

In [14]:
df_train.dtypes

[('Year', 'bigint'),
 ('Quarter', 'bigint'),
 ('Month', 'bigint'),
 ('DayofMonth', 'bigint'),
 ('DayOfWeek', 'bigint'),
 ('FlightDate', 'bigint'),
 ('Reporting_Airline', 'string'),
 ('Flight_Number_Reporting_Airline', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('CRSDepTime', 'bigint'),
 ('DepTimeBlk', 'string'),
 ('CRSArrTime', 'bigint'),
 ('ArrDel15', 'bigint'),
 ('CRSElapsedTime', 'double'),
 ('Distance', 'double'),
 ('DistanceGroup', 'bigint'),
 ('date', 'string'),
 ('dep_hour', 'bigint'),
 ('arr_hour', 'bigint'),
 ('dep_hour_minus2', 'bigint'),
 ('arr_hour_minus2', 'bigint'),
 ('origin_temp_f', 'double'),
 ('origin_dewpoint_f', 'double'),
 ('origin_humidity', 'double'),
 ('origin_feels_like_f', 'double'),
 ('origin_wind_kts', 'double'),
 ('origin_gust_kts', 'double'),
 ('origin_visibility', 'double'),
 ('origin_precip_in', 'double'),
 ('origin_wx_codes', 'string'),
 ('origin_is_rain', 'int'),
 ('origin_is_snow', 'int'),
 ('origin_is_fog', 'int'),
 ('origin_low_visibili

# Train Model

## Define Columns

In [15]:
target = 'ArrDel15'

drop_cols = [
    'FlightDate', 'date',
    'Flight_Number_Reporting_Airline',
    'origin_wx_codes', 'dest_wx_codes',
    target
]

categorical_cols = ['Reporting_Airline', 'Origin', 'Dest', 'DepTimeBlk']

# All numeric columns (everything else except target/drops/categoricals)
all_cols = [c for c, _ in df_train.dtypes]
numeric_cols = [c for c in all_cols if c not in drop_cols + categorical_cols]

## Check for NULLs and Cast Target

In [16]:
total = df_train.count()
null_row = df_train.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in numeric_cols
]).collect()[0].asDict()

nulls_found = {c: n for c, n in null_row.items() if n > 0}

if not nulls_found:
    print("✅ No nulls in any numeric column.")
else:
    print(f"⚠️ {len(nulls_found)} columns with nulls:")
    for c, n in sorted(nulls_found.items(), key=lambda x: -x[1]):
        print(f"  {c:<35} {n:>10,}  ({100*n/total:.2f}%)")

✅ No nulls in any numeric column.


In [ ]:
def prep(df):
    # Cast target to int (XGBoost requires integer labels for classification)
    df = df.withColumn(target, F.col(target).cast('int'))

    # Cast all numeric features to double for VectorAssembler consistency
    for c in numeric_cols:
        df = df.withColumn(c, F.col(c).cast('double'))

    return df

df_train = prep(df_train)
df_val   = prep(df_val)
df_test  = prep(df_test)

## Index Categoricals (Spark ML)

In [ ]:
# XGBoost handles integer-encoded categoricals well, and one-hot explodes dimensionality

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid='keep'   # unseen categories → extra bucket
    )
    for c in categorical_cols
]

## Assemble Feature Vector

In [ ]:
feature_cols = numeric_cols + [f"{c}_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
)

## Calculate Hyperparameter for Class Imbalance


In [20]:
counts = df_train_sampled.groupBy(target).count().collect()
counts_dict = {row[target]: row['count'] for row in counts}
scale_pos_weight = counts_dict[0] / counts_dict[1]
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

scale_pos_weight = 4.59


In [ ]:
# Year distribution
print("📅 Sampled train by year:")
df_train_sampled.groupBy('Year').count().orderBy('Year').show()

# Month distribution
print("📆 Sampled train by month:")
df_train_sampled.groupBy('Month').count().orderBy('Month').show()

# Compare delay rates: original vs sample
orig_rate = df_train.filter(F.col('ArrDel15')==1).count() / df_train.count()
samp_rate = counts_dict[1] / (counts_dict[0] + counts_dict[1])

print(f"\nOriginal delay rate: {orig_rate:.4f}")
print(f"Sample   delay rate: {samp_rate:.4f}")
print(f"Diff               : {abs(orig_rate - samp_rate)*100:.3f}%  ← should be < 0.1%")

## Configure Spark XGBoost Classifier

In [22]:
xgb_clf = SparkXGBClassifier(
    features_col='features',
    label_col=target,
    prediction_col='prediction',
    probability_col='probability',
    raw_prediction_col='rawPrediction',

    # Hyperparameters
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5.0,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    tree_method='hist',

    # Distributed training
    num_workers=1,           # set to # of Spark executors
    device='cuda',            # or 'cuda' for GPU
    early_stopping_rounds=50,
    validation_indicator_col='is_val'
)

## Combine Train + Val for Early Stopping

In [ ]:
# df_train_marked = df_train.withColumn('is_val', F.lit(False))
# df_val_marked   = df_val.withColumn('is_val', F.lit(True))
# df_trainval     = df_train_marked.unionByName(df_val_marked)

# Use SAMPLED val for early stopping (faster)
# Or use df_val_full if you have RAM headroom

df_train_marked = df_train_sampled.withColumn('is_val', F.lit(False))
df_val_marked   = df_val_sampled.withColumn('is_val', F.lit(True))   # or df_val_full

df_trainval = df_train_marked.unionByName(df_val_marked).repartition(8).cache()

print(f"Combined train+val: {df_trainval.count():,}")
print(f"Train rows: {df_trainval.filter(~F.col('is_val')).count():,}")
print(f"Val   rows: {df_trainval.filter( F.col('is_val')).count():,}")

## Build and Fit Pipeline

In [ ]:
pipeline = Pipeline(stages=indexers + [assembler, xgb_clf])

model = pipeline.fit(df_trainval)
print("✅ Training complete")

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.85, 'device': 'cuda', 'eval_metric': 'auc', 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 5.0, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': 4.593616448195367, 'subsample': 0.85, 'tree_method': 'hist', 'nthread': 1}
	train_call_kwargs_params: {'early_stopping_rounds': 50, 'verbose_eval': True, 'num_boost_round': 1000}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}


# Evaluate on Validation Set

In [ ]:
preds = model.transform(df_val_sampled)

# ROC-AUC & PR-AUC
auc_eval = BinaryClassificationEvaluator(
    labelCol=target, rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)
pr_eval = BinaryClassificationEvaluator(
    labelCol=target, rawPredictionCol='rawPrediction', metricName='areaUnderPR'
)

print(f"ROC-AUC: {auc_eval.evaluate(preds):.4f}")
print(f"PR-AUC : {pr_eval.evaluate(preds):.4f}")

# Accuracy / F1
acc_eval = MulticlassClassificationEvaluator(
    labelCol=target, predictionCol='prediction', metricName='accuracy'
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol=target, predictionCol='prediction', metricName='f1'
)
print(f"Accuracy: {acc_eval.evaluate(preds):.4f}")
print(f"F1      : {f1_eval.evaluate(preds):.4f}")

# Confusion matrix
preds.groupBy(target, 'prediction').count().orderBy(target, 'prediction').show()

## Threshold Tuning (Spark Side)

In [ ]:
from pyspark.sql.types import DoubleType

# Extract P(class=1) from probability vector
extract_p1 = F.udf(lambda v: float(v[1]), DoubleType())
preds_with_p = preds.withColumn('p1', extract_p1('probability'))

# Sweep thresholds
import numpy as np
results = []
for t in np.arange(0.2, 0.8, 0.05):
    tp = preds_with_p.filter((F.col('p1') >= t) & (F.col(target) == 1)).count()
    fp = preds_with_p.filter((F.col('p1') >= t) & (F.col(target) == 0)).count()
    fn = preds_with_p.filter((F.col('p1') <  t) & (F.col(target) == 1)).count()
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    results.append((round(t,2), prec, rec, f1))

for r in results:
    print(f"thr={r[0]:.2f} | P={r[1]:.3f} | R={r[2]:.3f} | F1={r[3]:.3f}")

## Feature Importance

In [ ]:
xgb_model = model.stages[-1]   # SparkXGBClassifierModel
booster = xgb_model.get_booster()

importance = booster.get_score(importance_type='gain')

# Map f0, f1, ... back to feature names
imp_named = sorted(
    [(feature_cols[int(k[1:])], v) for k, v in importance.items()],
    key=lambda x: -x[1]
)

for feat, score in imp_named[:20]:
    print(f"{feat:<35} {score:.2f}")

##